# Imports

In [10]:
# ------------------------
# Imports and Setup
# ------------------------
import os,glob,sys
import math
import random
import numpy as np
from scipy import stats
from scipy.io import savemat
from sklearn.cross_decomposition import CCA
from scipy.linalg import subspace_angles
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from pathlib import Path

# Load this for CTAE two region
from models.ctae import CoupledTransformerAutoencoderTwoRegions 

# Load this for CTAE multi region
from models.ctae import CoupledTransformerAutoencoderMultiRegion

from utils_correct import create_train_val_loaders, train_ctae_with_logging, safe_format, create_test_loader

# Functions

## data

In [2]:
# if already saved then just load:
# reload from npz (preferred) or .mat
def load_saved_dataset(regime_name, config, data_dir):
    npz_path = os.path.join(data_dir, f"{regime_name}.npz")

    if os.path.exists(npz_path):
        data = dict(np.load(npz_path, allow_pickle=True))
    else:
        raise FileNotFoundError(f"No saved file found for {regime_name}")

    return {"regime": regime_name, "config": config, "data": data}


def build_dataset_with_lag_lagged_stn_wo_vo(gpi_segs, stn_segs, lags=3):
    """
    Accepts either (N, C, T) or (N, T, C).
    Returns:
        X: (N, in_channels*(lags+1), T - lags)
        y: (N, out_channels*(lags+1), T - lags)
    """
    assert gpi_segs.shape[0] == stn_segs.shape[0], "Mismatched N between GPi and STN"

    # Standardize to (N, C, T)
    if gpi_segs.ndim != 3 or stn_segs.ndim != 3:
        raise ValueError("Inputs must be 3D arrays")

    # If we got (N, T, C), swap to (N, C, T)
    if gpi_segs.shape[1] > gpi_segs.shape[2]:
        gpi_segs = np.transpose(gpi_segs, (0, 2, 1))
        stn_segs = np.transpose(stn_segs, (0, 2, 1))

    X, y = [], []
    for gpi, stn in zip(gpi_segs, stn_segs):  # gpi, stn: (C, T)
        T = gpi.shape[1]
        if T <= lags:
            raise ValueError(f"time length T={T} must be > lags={lags}")

        # GPi lags (input)
        lagged_input = [gpi[:, i:T - lags + i] for i in range(lags + 1)]
        X_lag = np.concatenate(lagged_input, axis=0)  # (C*(lags+1), T-lags)

        # STN lags (output)
        lagged_stn = [stn[:, i:T - lags + i] for i in range(lags + 1)]
        y_lag = np.concatenate(lagged_stn, axis=0)    # (C*(lags+1), T-lags)

        X.append(X_lag)
        y.append(y_lag)

    return np.stack(X), np.stack(y)

## training

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

## eval

In [11]:
def compare_latent_performance(model_latents, gt_latents, model_name="Model"):
    """
    Compare learned latents to GT using CCA, subspace angle, and per-dim MSE.

    Parameters
    ----------
    model_latents : np.ndarray or torch.Tensor, shape (n_trials, T_model, D)
    gt_latents    : np.ndarray or torch.Tensor, shape (n_trials, D, T_gt)
    model_name    : str

    Returns
    -------
    results_df : pd.DataFrame  (per-dim CCA, angle, MSE)
    metrics    : dict           (mean_cca, mean_angle, mean_mse)
    """
    # Ensure numpy
    if hasattr(model_latents, "detach"):
        model_latents = model_latents.detach().cpu().numpy()
    if hasattr(gt_latents, "detach"):
        gt_latents = gt_latents.detach().cpu().numpy()

    n_trials, T_model, D = model_latents.shape

    # GT is (N, D, T_gt) -> (N, T_gt, D); then trim to match model T
    gt_latents = np.transpose(gt_latents, (0, 2, 1))  # (N, T_gt, D)
    gt_latents_trimmed = gt_latents[:, -T_model:, :]  # (N, T_model, D)

    # Flatten across trials/time -> (N*T, D)
    model_flat = model_latents.reshape(-1, D)
    gt_flat = gt_latents_trimmed.reshape(-1, D)

    # CCA (D comps)
    cca = CCA(n_components=D)
    model_cca, gt_cca = cca.fit_transform(model_flat, gt_flat)
    cca_corrs = [np.corrcoef(model_cca[:, i], gt_cca[:, i])[0, 1] for i in range(D)]
    mean_cca = float(np.mean(cca_corrs))

    # Subspace angles (principal angles between rank-D subspaces)
    # Build orthonormal bases via SVD of zero-mean data
    m_center = model_flat - model_flat.mean(axis=0, keepdims=True)
    g_center = gt_flat   - gt_flat.mean(axis=0, keepdims=True)
    _, _, VT_m = np.linalg.svd(m_center, full_matrices=False)
    _, _, VT_g = np.linalg.svd(g_center, full_matrices=False)
    Bm = VT_m[:D].T
    Bg = VT_g[:D].T
    angles_deg = np.degrees(subspace_angles(Bm, Bg))
    mean_angle = float(np.mean(angles_deg))

    # Per-dim MSE
    mse_per_dim = np.mean((model_flat - gt_flat) ** 2, axis=0)
    mean_mse = float(np.mean(mse_per_dim))

    results_df = pd.DataFrame({
        "CCA Correlation": cca_corrs,
        "Subspace Angle (deg)": angles_deg,
        "MSE": mse_per_dim
    })

    metrics = {
        "mean_cca": mean_cca,
        "mean_angle": mean_angle,
        "mean_mse": mean_mse
    }
    return results_df, metrics

def mean_cca(a: np.ndarray, b: np.ndarray, n_comp: int):
    """
    a,b: (N,T,D) -> flattened to (N*T, D). Returns mean CCA over D comps.
    """
    A = a.reshape(-1, a.shape[-1])
    B = b.reshape(-1, b.shape[-1])
    cca = CCA(n_components=n_comp)
    u, v = cca.fit_transform(A, B)
    corrs = [np.corrcoef(u[:, i], v[:, i])[0, 1] for i in range(n_comp)]
    return float(np.mean(corrs)), corrs

def _r2_per_channel(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-9):
    """
    y_*: (N, T, C) or (N*T, C). Returns mean R² over channels and trials/time.
    """
    if y_true.ndim == 3:
        N, T, C = y_true.shape
        yt = y_true.reshape(N*T, C)
        yp = y_pred.reshape(N*T, C)
    else:
        yt, yp = y_true, y_pred
        C = yt.shape[-1]
    # Center w.r.t. mean of y_true (per-channel)
    yt_c = yt - yt.mean(axis=0, keepdims=True)
    ss_tot = np.sum(yt_c**2, axis=0) + eps
    ss_res = np.sum((yt - yp)**2, axis=0)
    r2 = 1.0 - (ss_res / ss_tot)
    return float(np.mean(r2)), r2  # (mean over channels, vector per channel)

def _add_derived_columns(run_agg: pd.DataFrame) -> pd.DataFrame:
    # Cross vs shared-only consistency (smaller is better)
    d1 = (run_agg["r2_y_shared_only"] - run_agg["r2_y_from_x_shared"]).abs()
    d2 = (run_agg["r2_x_shared_only"] - run_agg["r2_x_from_y_shared"]).abs()
    run_agg["r2_shared_consistency"] = 0.5*(d1 + d2)

    # Leakage normalized by signal
    run_agg["leakage_ratio"] = (
        np.maximum(run_agg["leak_shared_vs_gt_private"], run_agg["leak_private_vs_gt_shared"]) /
        run_agg["mean_cca_shared"].clip(lower=1e-6)
    )

    # A simple, transparent Stage-1 ranking score
    run_agg["score_stage1"] = (
        0.45*run_agg["mean_cca_shared"] +
        0.20*run_agg["cca_shared12"] +
        0.20*((run_agg["r2_y_from_x_shared"] + run_agg["r2_x_from_y_shared"])/2.0) +
        0.10*((run_agg["r2_y_shared_only"] + run_agg["r2_x_shared_only"])/2.0) -
        0.05*run_agg["r2_shared_consistency"] -    # penalize big gaps
        0.05*run_agg["leakage_ratio"]               # penalize relative leakage
    )
    return run_agg

In [12]:
def compute_lfp_zscore_stats(x, eps=1e-8):
    # x: [N, T, C]
    mean = x.reshape(-1, x.shape[-1]).mean(dim=0)
    std = x.reshape(-1, x.shape[-1]).std(dim=0)
    return mean, std + eps


def apply_lfp_zscore(x, mean, std):
    return (x - mean) / std


def prepare_ctae_pair_from_tensors(gpi, stn, gpi_mean=None, gpi_std=None, stn_mean=None, stn_std=None):
    """
    Returns:
        gpi_z, stn_z, data, stats
    """
    gpi = gpi.float()
    stn = stn.float()

    N = min(gpi.shape[0], stn.shape[0])
    T = min(gpi.shape[1], stn.shape[1])

    gpi = gpi[:N, :T, :]
    stn = stn[:N, :T, :]

    if gpi_mean is None or gpi_std is None:
        gpi_mean, gpi_std = compute_lfp_zscore_stats(gpi)

    if stn_mean is None or stn_std is None:
        stn_mean, stn_std = compute_lfp_zscore_stats(stn)

    gpi_z = apply_lfp_zscore(gpi, gpi_mean, gpi_std)
    stn_z = apply_lfp_zscore(stn, stn_mean, stn_std)

    data = torch.cat([gpi_z, stn_z], dim=-1)  # [N, T, Cgpi+Cstn]

    stats = {
        "gpi_mean": gpi_mean,
        "gpi_std": gpi_std,
        "stn_mean": stn_mean,
        "stn_std": stn_std,
    }

    return gpi_z, stn_z, data, stats


@torch.no_grad()
def extract_ctae_latents_from_test_set(model, reg1_tensor, reg2_tensor, device, batch_size=32):
    """
    CTAE latent extraction.

    reg1_tensor, reg2_tensor: [N, T, C]
    Returns: shared1, shared2, private1, private2 as numpy arrays [N, T, D]
    """
    model = model.to(device).eval()

    reg1_tensor, reg2_tensor, _, _ = prepare_ctae_pair_from_tensors(
        reg1_tensor,
        reg2_tensor
    )
    N = min(reg1_tensor.shape[0], reg2_tensor.shape[0])
    T = min(reg1_tensor.shape[1], reg2_tensor.shape[1])

    reg1_tensor = reg1_tensor[:N, :T, :].float()
    reg2_tensor = reg2_tensor[:N, :T, :].float()
    num_neurons1 = reg1_tensor.shape[-1]

    s1_all, s2_all, p1_all, p2_all = [], [], [], []

    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)

        x1 = reg1_tensor[start:end].to(device)
        x2 = reg2_tensor[start:end].to(device)
        x = torch.cat([x1, x2], dim=-1)  # [B, T, C1+C2]

        (
            x11_hat, x22_hat, x12_hat, x21_hat,
            shared1, shared2, private1, private2, z
        ) = model(x, num_neurons1=num_neurons1)

        # CTAE outputs are [T, B, D], convert to [B, T, D]
        s1_all.append(shared1.permute(1, 0, 2).detach().cpu())
        s2_all.append(shared2.permute(1, 0, 2).detach().cpu())
        p1_all.append(private1.permute(1, 0, 2).detach().cpu())
        p2_all.append(private2.permute(1, 0, 2).detach().cpu())

    return (
        torch.cat(s1_all, dim=0).numpy(),
        torch.cat(s2_all, dim=0).numpy(),
        torch.cat(p1_all, dim=0).numpy(),
        torch.cat(p2_all, dim=0).numpy(),
    )


@torch.no_grad()
def evaluate_ctae_one_model_on_one_dataset(
    model,
    data,
    lags,
    device,
    run_name,
    latent_save_dir,
    batch_size=32,
):
    """
    CTAE synthetic evaluator:
      1. builds lagged inputs
      2. extracts CTAE latents
      3. saves latents
      4. computes GT CCA/MSE/angle using your existing compare_latent_performance
      5. computes leakage CCA and shared-region CCA
      6. computes reconstruction/cross-reconstruction R2
    """

    # -------- build inputs exactly like training --------
    reg1_R, reg2_R = build_dataset_with_lag_lagged_stn_wo_vo(
        data["region1"], data["region2"], lags=lags
    )

    reg1_tensor = torch.tensor(reg1_R, dtype=torch.float32).permute(0, 2, 1).to(device)
    reg2_tensor = torch.tensor(reg2_R, dtype=torch.float32).permute(0, 2, 1).to(device)

    num_neurons1 = reg1_tensor.shape[-1]
    model = model.to(device).eval()

    # -------- extract latents [N,T,D] --------
    s1_np, s2_np, p1_np, p2_np = extract_ctae_latents_from_test_set(
        model, reg1_tensor, reg2_tensor, device, batch_size=batch_size
    )

    # -------- save latents --------
    os.makedirs(latent_save_dir, exist_ok=True)
    latent_npz_path = os.path.join(latent_save_dir, f"{run_name}_latents.npz")
    np.savez_compressed(
        latent_npz_path,
        shared_reg1=s1_np,
        shared_reg2=s2_np,
        private_reg1=p1_np,
        private_reg2=p2_np,
    )

    # -------- GT comparisons --------
    gt_s1 = data["gt_shared1"]
    gt_s2 = data["gt_shared2"]
    gt_p1 = data["gt_private1"]
    gt_p2 = data["gt_private2"]

    _, m_shared1 = compare_latent_performance(s1_np, gt_s1, model_name=f"{run_name}_Shared1")
    _, m_shared2 = compare_latent_performance(s2_np, gt_s2, model_name=f"{run_name}_Shared2")
    _, m_priv1   = compare_latent_performance(p1_np, gt_p1, model_name=f"{run_name}_Private1")
    _, m_priv2   = compare_latent_performance(p2_np, gt_p2, model_name=f"{run_name}_Private2")

    # -------- align GT time length to model time length --------
    Tm = s1_np.shape[1]
    gt_p1_T = np.transpose(gt_p1, (0, 2, 1))[:, -Tm:, :]
    gt_s1_T = np.transpose(gt_s1, (0, 2, 1))[:, -Tm:, :]
    gt_p2_T = np.transpose(gt_p2, (0, 2, 1))[:, -Tm:, :]
    gt_s2_T = np.transpose(gt_s2, (0, 2, 1))[:, -Tm:, :]

    # -------- leakage checks --------
    mean_cca_s1_vs_gtp1, _ = mean_cca(s1_np, gt_p1_T, n_comp=min(s1_np.shape[-1], gt_p1_T.shape[-1]))
    mean_cca_p1_vs_gts1, _ = mean_cca(p1_np, gt_s1_T, n_comp=min(p1_np.shape[-1], gt_s1_T.shape[-1]))
    mean_cca_s2_vs_gtp2, _ = mean_cca(s2_np, gt_p2_T, n_comp=min(s2_np.shape[-1], gt_p2_T.shape[-1]))
    mean_cca_p2_vs_gts2, _ = mean_cca(p2_np, gt_s2_T, n_comp=min(p2_np.shape[-1], gt_s2_T.shape[-1]))

    # CTAE has no explicit alignment module, so shared12 is raw shared CCA
    mean_cca_shared12, _ = mean_cca(
        s1_np, s2_np, n_comp=min(s1_np.shape[-1], s2_np.shape[-1])
    )

    # -------- reconstruction/cross-reconstruction R2 --------
    x_true_all, y_true_all = [], []
    x_full_all, y_full_all = [], []
    y_from_x_all, x_from_y_all = [], []

    # N = reg1_tensor.shape[0]
    reg1_tensor, reg2_tensor, _, _ = prepare_ctae_pair_from_tensors(
        reg1_tensor,
        reg2_tensor
    )
    
    N = min(reg1_tensor.shape[0], reg2_tensor.shape[0])
    T = min(reg1_tensor.shape[1], reg2_tensor.shape[1])

    reg1_tensor = reg1_tensor[:N, :T, :]
    reg2_tensor = reg2_tensor[:N, :T, :]
    

    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)

        x1 = reg1_tensor[start:end].to(device)
        x2 = reg2_tensor[start:end].to(device)
        x = torch.cat([x1, x2], dim=-1)

        (
            x11_hat, x22_hat, x12_hat, x21_hat,
            shared1, shared2, private1, private2, z
        ) = model(x, num_neurons1=num_neurons1)

        # recon outputs are [T,B,C], convert to [B,T,C]
        x_full = x11_hat.permute(1, 0, 2)
        y_full = x22_hat.permute(1, 0, 2)
        y_from_x_shared = x12_hat.permute(1, 0, 2)
        x_from_y_shared = x21_hat.permute(1, 0, 2)

        x_true_all.append(x1.detach().cpu().numpy())
        y_true_all.append(x2.detach().cpu().numpy())
        x_full_all.append(x_full.detach().cpu().numpy())
        y_full_all.append(y_full.detach().cpu().numpy())
        y_from_x_all.append(y_from_x_shared.detach().cpu().numpy())
        x_from_y_all.append(x_from_y_shared.detach().cpu().numpy())

    x_true = np.concatenate(x_true_all, axis=0)
    y_true = np.concatenate(y_true_all, axis=0)
    x_full_np = np.concatenate(x_full_all, axis=0)
    y_full_np = np.concatenate(y_full_all, axis=0)
    y_from_x_np = np.concatenate(y_from_x_all, axis=0)
    x_from_y_np = np.concatenate(x_from_y_all, axis=0)

    r2_x_full_mean, _ = _r2_per_channel(x_true, x_full_np)
    r2_y_full_mean, _ = _r2_per_channel(y_true, y_full_np)
    r2_y_from_x_mean, _ = _r2_per_channel(y_true, y_from_x_np)
    r2_x_from_y_mean, _ = _r2_per_channel(x_true, x_from_y_np)

    # CTAE forward does not directly return same-region shared-only/private-only recon.
    # Keep these as NaN for fair reporting.
    common = {
        "latents_path": latent_npz_path,

        "cca_leak_s1_vs_gtp1": mean_cca_s1_vs_gtp1,
        "cca_leak_p1_vs_gts1": mean_cca_p1_vs_gts1,
        "cca_leak_s2_vs_gtp2": mean_cca_s2_vs_gtp2,
        "cca_leak_p2_vs_gts2": mean_cca_p2_vs_gts2,

        "cca_shared12": mean_cca_shared12,

        "r2_x_full": r2_x_full_mean,
        "r2_y_full": r2_y_full_mean,
        "r2_x_shared_only": np.nan,
        "r2_x_private_only": np.nan,
        "r2_y_shared_only": np.nan,
        "r2_y_private_only": np.nan,
        "r2_y_from_x_shared": r2_y_from_x_mean,
        "r2_x_from_y_shared": r2_x_from_y_mean,
    }

    rows = [
        {"latent_type": "Shared1",  **m_shared1, **common},
        {"latent_type": "Shared2",  **m_shared2, **common},
        {"latent_type": "Private1", **m_priv1, **common},
        {"latent_type": "Private2", **m_priv2, **common},
    ]

    return rows

In [17]:
def evaluate_all_ctae_models_and_datasets(
    TRAINED_CTAE_MODELS,
    ALL_DATASETS,
    save_excel_path,
    device="cuda" if torch.cuda.is_available() else "cpu",
    lags=3,
    latent_save_dir=r"F:\comp_project\synthecticData\latents_eval_ctae",
    batch_size=32,
):
    all_rows = []
    regime_to_data = {d["regime"]: d["data"] for d in ALL_DATASETS}

    for item in TRAINED_CTAE_MODELS:
        regime = item["regime"]
        variant = item.get("variant", "CTAE")
        model = item["model"]
        seed_model = item.get("seed", np.nan)
        
        run_name = f"{regime}_{variant}_seed{seed_model}"
        print(run_name)
        
        if regime not in regime_to_data:
            print(f"[WARN] Regime {regime} not found; skipping.")
            continue

        try:
            rows = evaluate_ctae_one_model_on_one_dataset(
                model=model,
                data=regime_to_data[regime],
                lags=lags,
                device=device,
                run_name=run_name,
                latent_save_dir=latent_save_dir,
                batch_size=batch_size,
            )

            for r in rows:
                r.update({
                    "regime": regime,
                    "variant": variant,
                    "seed_label": seed_model,
                    "model": "CTAE",
                })

            all_rows.extend(rows)
            print(f"[OK] {run_name}: evaluated and latents saved.")

        except Exception as e:
            print(f"[ERROR] {run_name}: {e}")

    if not all_rows:
        print("[WARN] No evaluation rows produced.")
        return pd.DataFrame()

    summary_df = pd.DataFrame(all_rows)

    meta_cols = ["model", "regime", "variant", "seed_label", "latent_type", "latents_path"]
    metric_cols = [c for c in summary_df.columns if c not in meta_cols]
    summary_df = summary_df[meta_cols + metric_cols]

    def _agg_run(g):
        shared = g[g["latent_type"].isin(["Shared1", "Shared2"])]
        priv = g[g["latent_type"].isin(["Private1", "Private2"])]

        out = {
            "mean_cca_shared": shared["mean_cca"].mean(),
            "mean_cca_private": priv["mean_cca"].mean(),
            "mean_angle_shared": shared["mean_angle"].mean(),
            "mean_angle_private": priv["mean_angle"].mean(),
            "mean_mse_shared": shared["mean_mse"].mean(),
            "mean_mse_private": priv["mean_mse"].mean(),
        }

        def first(col):
            s = g[col].dropna()
            return s.iloc[0] if len(s) else np.nan

        for c in [
            "cca_shared12",
            "r2_x_full", "r2_y_full",
            "r2_x_shared_only", "r2_y_shared_only",
            "r2_x_private_only", "r2_y_private_only",
            "r2_y_from_x_shared", "r2_x_from_y_shared",
        ]:
            if c in g.columns:
                out[c] = first(c)

        out["leak_shared_vs_gt_private"] = np.nanmean([
            g["cca_leak_s1_vs_gtp1"].mean(),
            g["cca_leak_s2_vs_gtp2"].mean(),
        ])
        out["leak_private_vs_gt_shared"] = np.nanmean([
            g["cca_leak_p1_vs_gts1"].mean(),
            g["cca_leak_p2_vs_gts2"].mean(),
        ])

        shared_cross_r2 = np.nanmean([
            out.get("r2_y_from_x_shared", np.nan),
            out.get("r2_x_from_y_shared", np.nan),
        ])
        leak_pen = np.nanmean([
            out.get("leak_shared_vs_gt_private", np.nan),
            out.get("leak_private_vs_gt_shared", np.nan),
        ])

        out["composite_score"] = (
            0.45 * out["mean_cca_shared"]
            + 0.25 * out.get("cca_shared12", np.nan)
            + 0.20 * shared_cross_r2
            - 0.10 * (leak_pen if not np.isnan(leak_pen) else 0.0)
        )

        return pd.Series(out)

    run_agg = (
        summary_df
        .groupby(["model", "regime", "variant", "seed_label"], as_index=False)
        .apply(_agg_run)
        .reset_index(drop=True)
    )

    if "_add_derived_columns" in globals():
        run_agg = _add_derived_columns(run_agg)

    os.makedirs(os.path.dirname(save_excel_path), exist_ok=True)

    with pd.ExcelWriter(save_excel_path, engine="openpyxl") as writer:
        summary_df.to_excel(writer, index=False, sheet_name="summary_all")
        run_agg.to_excel(writer, index=False, sheet_name="run_agg")

        for lt in ["Shared1", "Shared2", "Private1", "Private2"]:
            sub = summary_df[summary_df["latent_type"] == lt]
            if len(sub) > 0:
                sub.to_excel(writer, index=False, sheet_name=lt)

    print(f"[DONE] Wrote CTAE evaluation summary to: {save_excel_path}")
    return summary_df, run_agg

# Usage

## setup

In [4]:

# Your data are LFP windows, not binned spikes
fs = 500          # after downsampling
window_sec = 0.5
bin_size = 1 / fs # only used if you need a time vector

# Transformer settings: start small
nhead = 1
num_layers = 2

learning_rate = 1e-4

# Start conservative
lambda_ortho = 1e-3
lambda_alignment = 0.05
lambda_recons2 = 1
lambda_shared = 1

warm_up_ortho = 20

# Positional encoding
pe = True
pe_learn = False

# Your windows are ~247 or 250 timepoints
max_len = 300

batch_size = 8   # or 16 if GPU memory allows
num_epochs = 300 # start with 300, not 5000

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


cuda:0
NVIDIA RTX A4000


## data

In [5]:
data_save_dir = r"F:\comp_project\synthecticData\data"#your target directory to save datasets

DATA_PRESETS = {
    # 1) Linear-ish baseline (DLAG-friendly; sanity check)
    "S0_linear_easy": dict(
        n_trials=100, T=250, fs=500, shared_dim=3, private_dim=3,
        bipolarize=True,
        spatial_mixing="none",
        nonlin_mode="static",
        shared_variant="identity",
        add_one_over_f=False, ar_stage="none",
        sensor_noise=0.01,
        add_row_common=False, add_common_mode=False,
        seed=701,
    ),

    # 2) Nonlinear mapping (region-mismatched warp + gain+bilinear), no hetero noise
    #    DLAG starts to degrade; SPIRE should hold
    "S1_warp_gainbilin": dict(
        n_trials=100, T=250, fs=500, shared_dim=3, private_dim=3,
        bipolarize=True,
        spatial_mixing="random",              # or "gaussian" if you want harsher bipolar effects
        nonlin_mode="gain+bilinear", gain_g=0.8, bilinear_beta=0.8,
        interaction_strength=0.2,
        shared_variant="identity",
        shared_warp="region_mismatch",
        add_one_over_f=True, one_over_f_strength=0.2,
        ar_stage="latents", ar1_rho=0.3,
        sensor_noise=0.02,
        add_row_common=False, add_common_mode=False,
        seed=702,
    ),


    # 2) Nonlinear mapping (region-mismatched warp + gain+bilinear), no hetero noise
    #    DLAG starts to degrade; SPIRE should hold
    "D3_timevary_delay": dict(
        n_trials=100, T=250, fs=500, shared_dim=3, private_dim=3,
        bipolarize=True,
        spatial_mixing="random",              # or "gaussian" if you want harsher bipolar effects
        nonlin_mode="gain+bilinear", gain_g=0.8, bilinear_beta=0.8,
        interaction_strength=0.2,
        shared_variant="identity",
        shared_warp="region_mismatch",
        timevary_delay=True, tvd_amplitude=3, tvd_cycles=1.0,   # <-- NEW
        add_one_over_f=True, one_over_f_strength=0.2,
        ar_stage="latents", ar1_rho=0.3,
        sensor_noise=0.02,
        add_row_common=False, add_common_mode=False,
        seed=702,
    ),
    }

# rebuild ALL_DATASETS
ALL_DATASETS=[]
for regime_name, config in DATA_PRESETS.items():
    ALL_DATASETS.append(load_saved_dataset(regime_name, config, data_save_dir))


data_save_dir2 = r"F:\comp_project\synthecticData\dataT"#your target directory to save datasets

DATA_SETS = {
    "D3_sharedLow_privateHigh": dict(
    n_trials=100, T=250, fs=500, shared_dim=3, private_dim=3,
    bipolarize=True,
    spatial_mixing="random",
    nonlin_mode="gain+bilinear", gain_g=0.8, bilinear_beta=0.8,
    interaction_strength=0.2,

    shared_variant="identity",
    shared_warp="region_mismatch",
    timevary_delay=False,

    shared_freq_bases=[6, 10, 14],
    private_freq_bases=[22, 30, 38],
    shared_amp=1.0,
    private_amp=1.0,
    shared_add_one_over_f=False,
    private_add_one_over_f=False,

    ar_stage="latents", ar1_rho=0.3,
    sensor_noise=0.02,
    add_row_common=False, add_common_mode=False,
    seed=703,
),
    "D4_sharedHigh_privateLow": dict(
    n_trials=100, T=250, fs=500, shared_dim=3, private_dim=3,
    bipolarize=True,
    spatial_mixing="random",
    nonlin_mode="gain+bilinear", gain_g=0.8, bilinear_beta=0.8,
    interaction_strength=0.2,

    shared_variant="identity",
    shared_warp="region_mismatch",
    timevary_delay=False,

    shared_freq_bases=[22, 30, 38],
    private_freq_bases=[6, 10, 14],
    shared_amp=1.0,
    private_amp=1.2,   # optional: make slow private slightly larger to make the test harder
    shared_add_one_over_f=False,
    private_add_one_over_f=False,

    ar_stage="latents", ar1_rho=0.3,
    sensor_noise=0.02,
    add_row_common=False, add_common_mode=False,
    seed=704,
),
    "D5_sharedHigh_privateLow_strongPrivate": dict(
    n_trials=100, T=250, fs=500, shared_dim=3, private_dim=3,
    bipolarize=True,
    spatial_mixing="random",
    nonlin_mode="gain+bilinear", gain_g=0.8, bilinear_beta=0.8,
    interaction_strength=0.2,

    shared_variant="identity",
    shared_warp="region_mismatch",
    timevary_delay=False,

    shared_freq_bases=[24, 32, 40],
    private_freq_bases=[4, 7, 10],
    shared_amp=1.0,
    private_amp=1.5,
    shared_add_one_over_f=False,
    private_add_one_over_f=True,
    one_over_f_strength=0.15,

    ar_stage="latents", ar1_rho=0.3,
    sensor_noise=0.02,
    add_row_common=False, add_common_mode=False,
    seed=705,
),

}

for regime_name, config in DATA_SETS.items():
    ALL_DATASETS.append(load_saved_dataset(regime_name, config, data_save_dir2))

# quick sanity check
print(len(ALL_DATASETS), [d["regime"] for d in ALL_DATASETS])

6 ['S0_linear_easy', 'S1_warp_gainbilin', 'D3_timevary_delay', 'D3_sharedLow_privateHigh', 'D4_sharedHigh_privateLow', 'D5_sharedHigh_privateLow_strongPrivate']


## training

In [19]:
num_epochs = 500 # start with 300, not 5000
patience = 50

seed_list = [701]
TRAINED_MODELS_ROOT = r"F:\comp_project\ctae_models_synth"
os.makedirs(TRAINED_MODELS_ROOT, exist_ok=True)
LAGS= 0

shared_latent_dim = 3
r2_specific_dim = r1_specific_dim =3

# Loop over datasets and ablation variants
TRAINED_MODELS2 = []
for seed_model in seed_list:
    print(seed_model)
    for dataset in ALL_DATASETS:
        regime = dataset["regime"]
        data = dataset["data"]

        # unpack data
        region1_data, region2_data = data["region1"], data["region2"] #N,T,C

        # helper builds lag-augmented features and returns (N, C*lags+1, T-lags)
        # reg1_R, reg2_R = build_dataset_with_lag(region1_data, region2_data, lags=LAGS)
        print("before aug GPi:", region1_data.shape)
        print("STN:", region2_data.shape)
        reg1_R, reg2_R = build_dataset_with_lag_lagged_stn_wo_vo(region1_data, region2_data, lags=LAGS)
        # print(np.shape(reg1_R))

        # --- Convert to tensors and permute to (N, T, C) ---
        reg1_tensor = torch.tensor(reg1_R, dtype=torch.float32).permute(0, 2, 1)
        reg2_tensor = torch.tensor(reg2_R, dtype=torch.float32).permute(0, 2, 1)

        # use all as train (adjust if you want a split) we follow what DLAG did in demo
        reg1_train, reg2_train = reg1_tensor, reg2_tensor
        reg1_test, reg2_test = reg1_tensor, reg2_tensor

        print("GPi:", reg1_train.shape)
        print("STN:", reg2_train.shape)

        # Make sure same number of windows and timepoints
        N = min(reg1_train.shape[0], reg2_train.shape[0])
        T = min(reg1_train.shape[1], reg2_train.shape[1])

        reg1_train = reg1_train[:N, :T, :]
        reg2_train = reg2_train[:N, :T, :]

        # Optional but recommended: z-score per channel over all windows/time
        def zscore_lfp_tensor(x, eps=1e-8):
            # x: [N, T, C]
            mean = x.reshape(-1, x.shape[-1]).mean(dim=0)
            std = x.reshape(-1, x.shape[-1]).std(dim=0)
            return (x - mean) / (std + eps)

        reg1_train = zscore_lfp_tensor(reg1_train)
        reg2_train = zscore_lfp_tensor(reg2_train)

        data1 = reg1_train.numpy()
        data2 = reg2_train.numpy()

        # CTAE expects concatenated regions along channel dimension: [N, T, Cgpi + Cstn]
        data = np.concatenate((data1, data2), axis=-1).astype(np.float32)

        input_dim1 = data1.shape[-1]
        input_dim2 = data2.shape[-1]
        num_neurons1 = input_dim1
        num_timeframes = data.shape[1]

        time = np.arange(num_timeframes) * bin_size

        print("Final CTAE input:", data.shape)
        print("GPi channels:", input_dim1)
        print("STN channels:", input_dim2)
        print("Timeframes:", num_timeframes)
        
        ### 
        # ------------------------
        # Model save path
        # ------------------------
        model_path = (
            f"{TRAINED_MODELS_ROOT}/ctae_synth_{regime}"
            f"_bs{batch_size}"
            f"_lr{safe_format(learning_rate)}"
            f"_L{num_layers}"
            f"_r1-{r1_specific_dim}_r2-{r2_specific_dim}"
            f"_s{shared_latent_dim}"
            f"_pe{'T' if pe else 'F'}"
            f"_align{safe_format(lambda_alignment)}"
            f"_ortho{safe_format(lambda_ortho)}"
            f"_recons2-{safe_format(lambda_recons2)}"
            f"_warm{warm_up_ortho}"
            f"_seed{seed_model}"
            f"_ep{num_epochs}.pth"
        )
        print(f"\n=== {model_path} ===")

        hparam_dict = {
            "r1": r1_specific_dim,
            "r2": r2_specific_dim,
            "shared": shared_latent_dim,
            "nl": num_layers,
            "lambda_align": lambda_alignment,
            "lambda_ortho": lambda_ortho,
            "lr": learning_rate,
            "warm_up_ortho": warm_up_ortho,
            "batch_size": batch_size,
            "pe": pe,
        }

        hparam_str = "_".join([f"{k}-{safe_format(v)}" for k, v in hparam_dict.items()])
        print(model_path)

        train_loader, val_loader = create_train_val_loaders(
            data,
            batch_size=batch_size,
            train_ratio=0.8,
            shuffle=True,
            seed=0,
        )
        
        # ------------------------
        # Reproducibility
        # ------------------------

        torch.manual_seed(seed_model)
        np.random.seed(seed_model)
        random.seed(seed_model)

        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed_model)
            torch.cuda.manual_seed_all(seed_model)

        # ------------------------
        # Create model
        # ------------------------
        model = CoupledTransformerAutoencoderTwoRegions(
            input_dim1,
            input_dim2,
            r1_specific_dim,
            r2_specific_dim,
            shared_latent_dim,
            nhead,
            num_layers,
            num_layers,
            max_len,
            pe,
            pe_learn,
        )

        model = model.to(device)

        
        # ------------------------
        # Loss and optimizer
        # ------------------------
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

        metadata = {
            "regime": regime,
            "input_dim1": input_dim1,
            "input_dim2": input_dim2,
            "num_neurons1": num_neurons1,
            "num_timeframes": num_timeframes,
            "r1_specific_dim": r1_specific_dim,
            "r2_specific_dim": r2_specific_dim,
            "shared_latent_dim": shared_latent_dim,
            "nhead": nhead,
            "num_layers": num_layers,
            "max_len": max_len,
            "pe": pe,
            "pe_learn": pe_learn,
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "num_epochs": num_epochs,
            "lambda_alignment": lambda_alignment,
            "lambda_ortho": lambda_ortho,
            "lambda_recons2": lambda_recons2,
            "warm_up_ortho": warm_up_ortho,
        }

        bundle_path = model_path.replace(".pth", "_bundle.pth")
        log_dir = model_path.replace(".pth", "_tb")

        history,model = train_ctae_with_logging(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            num_epochs=num_epochs,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            num_neurons1=num_neurons1,
            model_path=model_path,
            lambda_alignment=lambda_alignment,
            lambda_ortho=lambda_ortho,
            warm_up_ortho=warm_up_ortho,
            lambda_recons2=lambda_recons2,
            early_stopping=True,
            patience=patience,
            min_delta=1e-4,
            start_epoch=150,
            log_dir=log_dir,
            bundle_path=bundle_path,
            metadata=metadata,
        )

        TRAINED_MODELS2.append({
            "regime": regime,
            "model_path": model_path,
            "model": model,
            "seed":seed_model
        })

701
before aug GPi: (100, 250, 8)
STN: (100, 250, 8)
GPi: torch.Size([100, 250, 8])
STN: torch.Size([100, 250, 8])
Final CTAE input: (100, 250, 16)
GPi channels: 8
STN channels: 8
Timeframes: 250

=== F:\comp_project\ctae_models_synth/ctae_synth_S0_linear_easy_bs8_lr0.0001_L2_r1-3_r2-3_s3_peT_align0.05_ortho0.001_recons2-1_warm20_seed701_ep500.pth ===
F:\comp_project\ctae_models_synth/ctae_synth_S0_linear_easy_bs8_lr0.0001_L2_r1-3_r2-3_s3_peT_align0.05_ortho0.001_recons2-1_warm20_seed701_ep500.pth


c:\Users\Rahil\Documents\ctae-multiregion\models\ctae.py:110: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(


Saved best model | epoch 0 | val R2=-0.1400 | val loss=0.553580
Epoch 0000 | train loss=0.617664, val loss=0.553580, train R2=-0.2160, val R2=-0.1400, align=0.034086, ortho=2.849404
Saved best model | epoch 1 | val R2=-0.0800 | val loss=0.524915
Saved best model | epoch 2 | val R2=-0.0610 | val loss=0.516014
Saved best model | epoch 3 | val R2=-0.0503 | val loss=0.511052
Saved best model | epoch 4 | val R2=-0.0433 | val loss=0.507739
Saved best model | epoch 5 | val R2=-0.0379 | val loss=0.505180
Saved best model | epoch 6 | val R2=-0.0334 | val loss=0.502954
Saved best model | epoch 7 | val R2=-0.0290 | val loss=0.500764
Saved best model | epoch 8 | val R2=-0.0253 | val loss=0.498972
Saved best model | epoch 9 | val R2=-0.0221 | val loss=0.497415
Saved best model | epoch 10 | val R2=-0.0191 | val loss=0.495962
Epoch 0010 | train loss=0.519833, val loss=0.495962, train R2=-0.0327, val R2=-0.0191, align=0.006771, ortho=4.545117
Saved best model | epoch 11 | val R2=-0.0167 | val loss=0.4

In [ ]:
# seed_list = [701,702,703,704]
# TRAINED_MODELS_ROOT = r"F:\comp_project\ctae_models_synth"
# os.makedirs(TRAINED_MODELS_ROOT, exist_ok=True)
# LAGS= 0

# shared_latent_dim = 3
# r2_specific_dim = r1_specific_dim =3

# # Loop over datasets and ablation variants
# TRAINED_MODELS = []
# for seed_model in seed_list:
#     print(seed_model)
#     for dataset in ALL_DATASETS:
#         regime = dataset["regime"]
#         data = dataset["data"]

#         # unpack data
#         region1_data, region2_data = data["region1"], data["region2"] #N,T,C

#         # helper builds lag-augmented features and returns (N, C*lags+1, T-lags)
#         # reg1_R, reg2_R = build_dataset_with_lag(region1_data, region2_data, lags=LAGS)
#         print("before aug GPi:", region1_data.shape)
#         print("STN:", region2_data.shape)
#         reg1_R, reg2_R = build_dataset_with_lag_lagged_stn_wo_vo(region1_data, region2_data, lags=LAGS)
#         # print(np.shape(reg1_R))

#         # --- Convert to tensors and permute to (N, T, C) ---
#         reg1_tensor = torch.tensor(reg1_R, dtype=torch.float32).permute(0, 2, 1)
#         reg2_tensor = torch.tensor(reg2_R, dtype=torch.float32).permute(0, 2, 1)

#         # use all as train (adjust if you want a split) we follow what DLAG did in demo
#         reg1_train, reg2_train = reg1_tensor, reg2_tensor
#         reg1_test, reg2_test = reg1_tensor, reg2_tensor

#         print("GPi:", reg1_train.shape)
#         print("STN:", reg2_train.shape)

#         # Make sure same number of windows and timepoints
#         N = min(reg1_train.shape[0], reg2_train.shape[0])
#         T = min(reg1_train.shape[1], reg2_train.shape[1])

#         reg1_train = reg1_train[:N, :T, :]
#         reg2_train = reg2_train[:N, :T, :]

#         # Optional but recommended: z-score per channel over all windows/time
#         def zscore_lfp_tensor(x, eps=1e-8):
#             # x: [N, T, C]
#             mean = x.reshape(-1, x.shape[-1]).mean(dim=0)
#             std = x.reshape(-1, x.shape[-1]).std(dim=0)
#             return (x - mean) / (std + eps)

#         reg1_train = zscore_lfp_tensor(reg1_train)
#         reg2_train = zscore_lfp_tensor(reg2_train)

#         data1 = reg1_train.numpy()
#         data2 = reg2_train.numpy()

#         # CTAE expects concatenated regions along channel dimension: [N, T, Cgpi + Cstn]
#         data = np.concatenate((data1, data2), axis=-1).astype(np.float32)

#         input_dim1 = data1.shape[-1]
#         input_dim2 = data2.shape[-1]
#         num_neurons1 = input_dim1
#         num_timeframes = data.shape[1]

#         time = np.arange(num_timeframes) * bin_size

#         print("Final CTAE input:", data.shape)
#         print("GPi channels:", input_dim1)
#         print("STN channels:", input_dim2)
#         print("Timeframes:", num_timeframes)
        
#         ### 
#         # ------------------------
#         # Model save path
#         # ------------------------
#         model_path = (
#             f"{TRAINED_MODELS_ROOT}/ctae_synth_{regime}"
#             f"_bs{batch_size}"
#             f"_lr{safe_format(learning_rate)}"
#             f"_L{num_layers}"
#             f"_r1-{r1_specific_dim}_r2-{r2_specific_dim}"
#             f"_s{shared_latent_dim}"
#             f"_pe{'T' if pe else 'F'}"
#             f"_align{safe_format(lambda_alignment)}"
#             f"_ortho{safe_format(lambda_ortho)}"
#             f"_recons2-{safe_format(lambda_recons2)}"
#             f"_warm{warm_up_ortho}"
#             f"_seed{seed_model}"
#             f"_ep{num_epochs}.pth"
#         )
#         print(f"\n=== {model_path} ===")

#         hparam_dict = {
#             "r1": r1_specific_dim,
#             "r2": r2_specific_dim,
#             "shared": shared_latent_dim,
#             "nl": num_layers,
#             "lambda_align": lambda_alignment,
#             "lambda_ortho": lambda_ortho,
#             "lr": learning_rate,
#             "warm_up_ortho": warm_up_ortho,
#             "batch_size": batch_size,
#             "pe": pe,
#         }

#         hparam_str = "_".join([f"{k}-{safe_format(v)}" for k, v in hparam_dict.items()])
#         print(model_path)

#         train_loader, val_loader = create_train_val_loaders(
#             data,
#             batch_size=batch_size,
#             train_ratio=0.8,
#             shuffle=True,
#             seed=0,
#         )
        
#         # ------------------------
#         # Reproducibility
#         # ------------------------

#         torch.manual_seed(seed_model)
#         np.random.seed(seed_model)
#         random.seed(seed_model)

#         if torch.cuda.is_available():
#             torch.cuda.manual_seed(seed_model)
#             torch.cuda.manual_seed_all(seed_model)

#         # ------------------------
#         # Create model
#         # ------------------------
#         model = CoupledTransformerAutoencoderTwoRegions(
#             input_dim1,
#             input_dim2,
#             r1_specific_dim,
#             r2_specific_dim,
#             shared_latent_dim,
#             nhead,
#             num_layers,
#             num_layers,
#             max_len,
#             pe,
#             pe_learn,
#         )

#         model = model.to(device)

        
#         # ------------------------
#         # Loss and optimizer
#         # ------------------------
#         criterion = nn.MSELoss()
#         optimizer = optim.Adam(model.parameters(), lr=learning_rate)

#         metadata = {
#             "regime": regime,
#             "input_dim1": input_dim1,
#             "input_dim2": input_dim2,
#             "num_neurons1": num_neurons1,
#             "num_timeframes": num_timeframes,
#             "r1_specific_dim": r1_specific_dim,
#             "r2_specific_dim": r2_specific_dim,
#             "shared_latent_dim": shared_latent_dim,
#             "nhead": nhead,
#             "num_layers": num_layers,
#             "max_len": max_len,
#             "pe": pe,
#             "pe_learn": pe_learn,
#             "learning_rate": learning_rate,
#             "batch_size": batch_size,
#             "num_epochs": num_epochs,
#             "lambda_alignment": lambda_alignment,
#             "lambda_ortho": lambda_ortho,
#             "lambda_recons2": lambda_recons2,
#             "warm_up_ortho": warm_up_ortho,
#         }

#         bundle_path = model_path.replace(".pth", "_bundle.pth")
#         log_dir = model_path.replace(".pth", "_tb")

#         history,model = train_ctae_with_logging(
#             model=model,
#             train_loader=train_loader,
#             val_loader=val_loader,
#             num_epochs=num_epochs,
#             criterion=criterion,
#             optimizer=optimizer,
#             device=device,
#             num_neurons1=num_neurons1,
#             model_path=model_path,
#             lambda_alignment=lambda_alignment,
#             lambda_ortho=lambda_ortho,
#             warm_up_ortho=warm_up_ortho,
#             lambda_recons2=lambda_recons2,
#             early_stopping=True,
#             patience=30,
#             min_delta=1e-4,
#             start_epoch=80,
#             log_dir=log_dir,
#             bundle_path=bundle_path,
#             metadata=metadata,
#         )

#         TRAINED_MODELS.append({
#             "regime": regime,
#             "model_path": model_path,
#             "model": model,
#             "seed":seed_model
#         })

## eval

In [21]:
print(f"Evaluating ...")
results_df = evaluate_all_ctae_models_and_datasets(
    TRAINED_CTAE_MODELS=TRAINED_MODELS2,
    ALL_DATASETS=ALL_DATASETS,
    save_excel_path=r"F:\comp_project\synthecticData\evaluation\CTAE_synth_6Data_noLag_1seed_500E.xlsx",
    device="cuda" if torch.cuda.is_available() else "cpu",
    lags=LAGS,
    latent_save_dir=r"F:\comp_project\synthecticData\CTAE_6Data_noLag_4seeds_500E"
)
display(results_df.head())

Evaluating ...
S0_linear_easy_CTAE_seed701
[OK] S0_linear_easy_CTAE_seed701: evaluated and latents saved.
S1_warp_gainbilin_CTAE_seed701
[OK] S1_warp_gainbilin_CTAE_seed701: evaluated and latents saved.
D3_timevary_delay_CTAE_seed701
[OK] D3_timevary_delay_CTAE_seed701: evaluated and latents saved.
D3_sharedLow_privateHigh_CTAE_seed701
[OK] D3_sharedLow_privateHigh_CTAE_seed701: evaluated and latents saved.
D4_sharedHigh_privateLow_CTAE_seed701
[OK] D4_sharedHigh_privateLow_CTAE_seed701: evaluated and latents saved.
D5_sharedHigh_privateLow_strongPrivate_CTAE_seed701
[OK] D5_sharedHigh_privateLow_strongPrivate_CTAE_seed701: evaluated and latents saved.
[DONE] Wrote CTAE evaluation summary to: F:\comp_project\synthecticData\evaluation\CTAE_synth_6Data_noLag_1seed_500E.xlsx


AttributeError: 'tuple' object has no attribute 'head'